In [1]:
!pip install torchdiffeq -q
!pip install albumentations==1.3.1 -q
!pip install scipy -q
print('✅ All packages installed.')

✅ All packages installed.


In [2]:
# ─── CHANGE THIS to your actual folder path ───────────────────────
DATASET_PATH = 'Dataset_BUSI_with_GT'
# ──────────────────────────────────────────────────────────────────

import os
assert os.path.exists(DATASET_PATH), f'❌ Path not found: {DATASET_PATH}'
print(f'✅ Dataset found at: {DATASET_PATH}')

✅ Dataset found at: Dataset_BUSI_with_GT


In [3]:
!pip install torchdiffeq -q
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree
from scipy.ndimage import binary_erosion

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchdiffeq import odeint

import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Imports done. Device: {device}')

✅ Imports done. Device: cuda


In [4]:
# ── Training hyperparameters (exactly as in paper) ────────────────
BATCH_SIZE   = 12       # paper: batch size 12
TOTAL_EPOCHS = 50      # paper: 200 iterations
INIT_LR      = 0.03     # paper: initial lr 0.03
WEIGHT_DECAY = 5e-4     # paper: weight decay 5e-4
MOMENTUM     = 0.9      # paper: momentum 0.9
ALPHA        = 1.0      # CE loss weight   (paper: α=1)
BETA         = 1.0      # Dice loss weight (paper: β=1)
IMAGE_SIZE   = 512      # paper: resize all images to 512×512
NUM_CLASSES  = 2        # background + tumor
VAL_SPLIT    = 0.2      # paper: 80% train / 20% val
ODE_METHOD   = 'dopri5'  # fast; change to 'dopri5' for higher accuracy
SAVE_PATH    = 'unode_best.pth'

print('✅ Hyperparameters set.')

✅ Hyperparameters set.


In [5]:
def get_image_mask_pairs(root_dir):
    """
    Scans benign/ and malignant/ subfolders.
    Excludes normal/ as per paper.
    Returns list of (image_path, [mask_paths]) tuples.
    """
    pairs = []
    for category in ['benign', 'malignant']:
        folder = os.path.join(root_dir, category)
        if not os.path.exists(folder):
            print(f'⚠️  Not found: {folder}')
            continue
        images = sorted([
            f for f in glob(os.path.join(folder, '*.png'))
            if '_mask' not in os.path.basename(f)
        ])
        for img_path in images:
            base  = img_path.replace('.png', '')
            masks = sorted(glob(base + '_mask*.png'))
            if masks:
                pairs.append((img_path, masks))
    print(f'Total samples (benign + malignant): {len(pairs)}')
    return pairs


def merge_masks(mask_paths):
    """Merge multiple masks with logical OR (as stated in paper)."""
    merged = None
    for mp in mask_paths:
        m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        merged = m.astype(bool) if merged is None else (merged | m.astype(bool))
    return (merged.astype(np.uint8) * 255) if merged is not None else np.zeros((1,1), np.uint8)


class BUSIDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs     = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_paths = self.pairs[idx]

        # Load image → grayscale → 3-channel RGB
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

        # Merge and binarize mask
        mask = merge_masks(mask_paths)

        # Resize to 512×512
        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
        mask  = cv2.resize(mask,  (IMAGE_SIZE, IMAGE_SIZE),
                           interpolation=cv2.INTER_NEAREST)
        mask  = (mask > 127).astype(np.uint8)  # 0=background, 1=tumor

        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug['image']         # tensor [3,H,W] normalized
            mask  = aug['mask'].long()   # tensor [H,W] int64
        else:
            image = torch.from_numpy(image.transpose(2,0,1)).float() / 255.0
            mask  = torch.from_numpy(mask).long()

        return image, mask


train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.Affine(scale=(0.9,1.1),
             translate_percent={'x':(-0.1,0.1),'y':(-0.1,0.1)},
             rotate=(-15,15), shear=(-5,5), p=0.5),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])


# Build dataloaders
all_pairs = get_image_mask_pairs(DATASET_PATH)
train_pairs, val_pairs = train_test_split(
    all_pairs, test_size=VAL_SPLIT, random_state=42, shuffle=True
)
print(f'Train: {len(train_pairs)} | Val: {len(val_pairs)}')

train_loader = DataLoader(
    BUSIDataset(train_pairs, transform=train_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    BUSIDataset(val_pairs, transform=val_transform),
    batch_size=1, shuffle=False, num_workers=2, pin_memory=True
)

# Sanity check
imgs, masks = next(iter(train_loader))
print(f'Image batch: {imgs.shape}')   # [12, 3, 512, 512]
print(f'Mask batch:  {masks.shape}')  # [12, 512, 512]
print(f'Mask values: {masks.unique()}')  # [0, 1]
print('✅ Dataset ready.')

Total samples (benign + malignant): 647
Train: 517 | Val: 130
Image batch: torch.Size([12, 3, 512, 512])
Mask batch:  torch.Size([12, 512, 512])
Mask values: tensor([0, 1])
✅ Dataset ready.


In [6]:
# ============================================================
# CELL 6 (FIXED) — U-Node Model
# base_ch reduced so total params ≈ 2.09M (matches paper Table 1)
# Replace your entire Cell 6 with this
# ============================================================

import torch
import torch.nn as nn
from torchdiffeq import odeint


# ── ODE Function: dx/dt = f(x, t, θ) ─────────────────────────────
class ODEFunc(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm2d(channels),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(channels),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
        )
    def forward(self, t, x):
        return self.block(x)


# ── ODE Block: integrates from t=0 to t=1 ────────────────────────
class ODEBlock(nn.Module):
    def __init__(self, channels, method='euler', tol=1e-3):
        super().__init__()
        self.odefunc = ODEFunc(channels)
        self.method  = method
        self.tol     = tol
        self.register_buffer('t', torch.tensor([0.0, 1.0]))
    def forward(self, x):
        out = odeint(self.odefunc, x, self.t,
                     method=self.method, rtol=self.tol, atol=self.tol)
        return out[1]


# ── ODE Unit = 1×1 entry conv + ODE block + 1×1 exit conv ────────
# This is what the paper calls an "ODE block" in Fig. 1
class ODEUnit(nn.Module):
    def __init__(self, in_ch, out_ch, method='euler'):
        super().__init__()
        self.entry = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        self.ode  = ODEBlock(out_ch, method=method)
        self.exit = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.exit(self.ode(self.entry(x)))


# ── U-Node ────────────────────────────────────────────────────────
# KEY FIX: base_ch=16 gives channels [16,32,64,128,256]
# This matches the paper's ~2.09M parameter count.
# base_ch=64 gives [64,128,256,512,1024] = ~34M params (wrong)
class UNode(nn.Module):
    def __init__(self, in_channels=3, num_classes=2,
                 base_ch=16, method='euler'):
        super().__init__()
        ch = [base_ch,
              base_ch * 2,
              base_ch * 4,
              base_ch * 8,
              base_ch * 16]
        # [16, 32, 64, 128, 256]

        # Encoder
        self.enc1 = ODEUnit(in_channels, ch[0], method)
        self.enc2 = ODEUnit(ch[0],       ch[1], method)
        self.enc3 = ODEUnit(ch[1],       ch[2], method)
        self.enc4 = ODEUnit(ch[2],       ch[3], method)

        # Downsample / Upsample (Interpolate as in paper Fig. 1)
        self.down = nn.Upsample(scale_factor=0.5, mode='bilinear',
                                align_corners=False)
        self.up   = nn.Upsample(scale_factor=2.0, mode='bilinear',
                                align_corners=False)

        # Bottleneck
        self.bottleneck = ODEUnit(ch[3], ch[4], method)

        # Decoder — input channels = upsampled + skip connection concat
        self.dec4 = ODEUnit(ch[4] + ch[3], ch[3], method)
        self.dec3 = ODEUnit(ch[3] + ch[2], ch[2], method)
        self.dec2 = ODEUnit(ch[2] + ch[1], ch[1], method)
        self.dec1 = ODEUnit(ch[1] + ch[0], ch[0], method)

        # Output
        self.out_conv = nn.Conv2d(ch[0], num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.down(e1))
        e3 = self.enc3(self.down(e2))
        e4 = self.enc4(self.down(e3))
        # Bottleneck
        b  = self.bottleneck(self.down(e4))
        # Decoder
        d4 = self.dec4(torch.cat([self.up(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], dim=1))
        return self.out_conv(d1)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ── Verify param count matches paper ─────────────────────────────
model = UNode(in_channels=3, num_classes=2,
              base_ch=16, method=ODE_METHOD).to(device)

# Use small dummy input to avoid OOM during check
with torch.no_grad():
    test_out = model(torch.randn(1, 3, 64, 64).to(device))

params_m = count_params(model) / 1e6
print(f'Output shape : {test_out.shape}')
print(f'Parameters   : {params_m:.4f}M')

if abs(params_m - 2.09) < 0.5:
    print('✅ Param count matches paper (~2.09M). Proceed to Cell 7.')
else:
    print(f'⚠️  Param count {params_m:.2f}M differs from paper 2.09M.')
    print('   Adjust base_ch value until it matches.')


Output shape : torch.Size([1, 2, 64, 64])
Parameters   : 2.1854M
✅ Param count matches paper (~2.09M). Proceed to Cell 7.


In [7]:
class DiceLoss(nn.Module):
    """Soft Dice loss — Milletari et al. V-Net 2016 (paper eq. 9)"""
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        # logits:  [B, 2, H, W]
        # targets: [B, H, W]  int64
        probs  = F.softmax(logits, dim=1)[:, 1]  # tumor probability [B,H,W]
        flat_p = probs.reshape(-1)
        flat_t = targets.float().reshape(-1)
        inter  = (flat_p * flat_t).sum()
        return 1.0 - (2.0*inter + self.smooth) / \
                     (flat_p.sum() + flat_t.sum() + self.smooth)


class CombinedLoss(nn.Module):
    """L = α*CE + β*Dice  (paper eq. 10, α=β=1)"""
    def __init__(self, alpha=1.0, beta=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.ce    = nn.CrossEntropyLoss()
        self.dice  = DiceLoss()

    def forward(self, logits, targets):
        return self.alpha * self.ce(logits, targets) + \
               self.beta  * self.dice(logits, targets)


criterion = CombinedLoss(ALPHA, BETA).to(device)
print('✅ Loss functions ready.')

✅ Loss functions ready.


In [8]:
# SGD exactly as in paper
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=INIT_LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY
)

# Polynomial decay: lr = init_lr × (1 - epoch/total_epochs)^0.9  (paper eq. 18)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda epoch: (1.0 - epoch / TOTAL_EPOCHS) ** 0.9
)

print('✅ Optimizer and scheduler ready.')

✅ Optimizer and scheduler ready.


In [9]:
train_losses = []
val_losses   = []
best_val_loss = float('inf')

for epoch in range(1, 50 + 1):

    # ── Train ───────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # ── Validate ────────────────────────────────────────────
    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            val_loss_sum += criterion(logits, masks).item()
    val_loss = val_loss_sum / len(val_loader)
    val_losses.append(val_loss)

    scheduler.step()

    # Save best model checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_PATH)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{TOTAL_EPOCHS} | '
              f'Train: {train_loss:.4f} | '
              f'Val: {val_loss:.4f} | '
              f'LR: {scheduler.get_last_lr()[0]:.5f}')

print(f'\n✅ Training complete. Best val loss: {best_val_loss:.4f}')
print(f'   Model saved → {SAVE_PATH}')

OutOfMemoryError: CUDA out of memory. Tried to allocate 960.00 MiB. GPU 0 has a total capacity of 79.15 GiB of which 216.19 MiB is free. Process 78299 has 848.00 MiB memory in use. Process 77474 has 450.00 MiB memory in use. Process 52583 has 77.65 GiB memory in use. Of the allocated memory 75.93 GiB is allocated by PyTorch, and 1.23 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='U-Node Train')
plt.title('(a) Training Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(val_losses, label='U-Node Val', color='orange')
plt.title('(b) Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150)
plt.show()
print('✅ Loss curves saved.')

In [ ]:
# All 7 metrics as defined in paper Section 3.2 (equations 11–17)

def compute_tp_fp_tn_fn(pred, gt):
    TP = int(np.logical_and( pred,  gt).sum())
    FP = int(np.logical_and( pred, ~gt).sum())
    TN = int(np.logical_and(~pred, ~gt).sum())
    FN = int(np.logical_and(~pred,  gt).sum())
    return TP, FP, TN, FN

def dice_metric(TP, FP, FN):
    d = 2*TP + FP + FN
    return (2*TP / d * 100) if d > 0 else 0.0

def jaccard_metric(TP, FP, FN):
    d = TP + FP + FN
    return (TP / d * 100) if d > 0 else 0.0

def precision_metric(TP, FP):
    d = TP + FP
    return (TP / d * 100) if d > 0 else 0.0

def sensitivity_metric(TP, FN):
    d = TP + FN
    return (TP / d * 100) if d > 0 else 0.0

def specificity_metric(TN, FP):
    d = FP + TN
    return (TN / d * 100) if d > 0 else 0.0

def get_surface_points(mask):
    if not mask.any():
        return None
    surface = mask & ~binary_erosion(mask)
    return np.argwhere(surface)

def hausdorff_distance(pred, gt):
    """Eq. 17 — lower is better. Returns None if pred is empty."""
    s_pred = get_surface_points(pred)
    s_gt   = get_surface_points(gt)
    if s_pred is None or s_gt is None:
        return None
    from scipy.spatial.distance import directed_hausdorff
    return max(directed_hausdorff(s_pred, s_gt)[0],
               directed_hausdorff(s_gt, s_pred)[0])

def assd_metric(pred, gt):
    """Eq. 16 — lower is better. Returns None if pred is empty."""
    s_pred = get_surface_points(pred)
    s_gt   = get_surface_points(gt)
    if s_pred is None or s_gt is None:
        return None
    tree_gt   = cKDTree(s_gt)
    tree_pred = cKDTree(s_pred)
    d_pg, _   = tree_gt.query(s_pred, k=1)
    d_gp, _   = tree_pred.query(s_gt, k=1)
    return (d_pg.sum() + d_gp.sum()) / (len(s_pred) + len(s_gt))

print('✅ Metric functions ready.')

In [ ]:
# Load the best saved checkpoint
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()

all_dice, all_jaccard   = [], []
all_precision           = []
all_sensitivity         = []
all_specificity         = []
all_assd, all_hd        = [], []
na_count                = 0

with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc='Evaluating'):
        imgs  = imgs.to(device)
        gt    = masks.numpy().squeeze().astype(bool)  # [H,W] bool

        logits = model(imgs)                           # [1,2,H,W]
        pred   = F.softmax(logits, dim=1)[0, 1].cpu().numpy() > 0.5  # [H,W] bool

        TP, FP, TN, FN = compute_tp_fp_tn_fn(pred, gt)

        all_dice.append(dice_metric(TP, FP, FN))
        all_jaccard.append(jaccard_metric(TP, FP, FN))
        all_precision.append(precision_metric(TP, FP))
        all_sensitivity.append(sensitivity_metric(TP, FN))
        all_specificity.append(specificity_metric(TN, FP))

        hd_val   = hausdorff_distance(pred, gt)
        assd_val = assd_metric(pred, gt)

        if hd_val   is not None: all_hd.append(hd_val)
        else: na_count += 1
        if assd_val is not None: all_assd.append(assd_val)


def mean(lst): return np.mean(lst) if lst else float('nan')

print('\n' + '='*62)
print(f'{"METRIC":<22} {"YOUR MODEL":>10}  {"PAPER (U-Node)":>14}')
print('='*62)
print(f'{"Dice (%) ↑":<22} {mean(all_dice):>10.2f}  {"76.40":>14}')
print(f'{"ASSD ↓":<22} {mean(all_assd):>10.2f}  {"20.46":>14}')
print(f'{"Jaccard (%) ↑":<22} {mean(all_jaccard):>10.2f}  {"68.39":>14}')
print(f'{"Precision (%) ↑":<22} {mean(all_precision):>10.2f}  {"78.48":>14}')
print(f'{"Sensitivity (%) ↑":<22} {mean(all_sensitivity):>10.2f}  {"79.59":>14}')
print(f'{"Specificity (%) ↑":<22} {mean(all_specificity):>10.2f}  {"97.56":>14}')
print(f'{"HD ↓":<22} {mean(all_hd):>10.2f}  {"76.62":>14}')
print(f'{"Params (M) ↓":<22} {count_params(model)/1e6:>10.4f}  {"2.09":>14}')
print('='*62)
if na_count > 0:
    print(f'⚠️  Empty predictions (N/A cases): {na_count} — excluded from ASSD/HD average')
print('\n✅ Evaluation complete.')